# IPPO + Muon：Colab 快速吞吐测试

在 Colab 选择 L4、G4 或 A100 GPU 后运行全部单元。保持 LocalServer 的完整斗地主、v1.3 模型、Muon、4 PPO epoch、OMP6、rollout256/collect224/mini3 和 Inductor 参数。测试 40 轮；前 8 轮作为预热，在第 9 轮完成边界后计时，共比较后续 31 个完整训练轮次。

不保存 checkpoint 或训练模型。仅输出吞吐、显存及版本信息。采用 Colab 自带的 CUDA PyTorch；软件版本、CPU 和内存差异会影响比较。500ms GPU 采样为观察峰值，不是精确分配峰值。单个 seed 的快速测试不能衡量棋力或长期收敛。

## 2026-10-10 Colab 实测记录

IPPO + Muon、完整斗地主、v1.3 模型、4 PPO epoch、rollout256 / collect224 / mini3、OMP6、Inductor，seed=0。每卡运行 40 轮；排除启动和预热后，计量 31 个完整轮次（6,944 对局）。

| Colab 类型 | 对局/秒 | 游戏步/秒 | 秒/轮 | 显存观察峰值 GiB |
|---|---:|---:|---:|---:|
| L4 | 47.24 | 2944 | 4.742 | 12.08 |
| G4 | 221.25 | 13666 | 1.012 | 12.49 |
| A100 | 113.84 | 7066 | 1.968 | 12.35 |

三卡均为 Python 3.13.15、PyTorch 2.11.0+cu130、CUDA 13.0、驱动 580.82.07。实际硬件与冷启动耗时：

| 类型 | 实际 GPU | CPU（可见 vCPU） | 主机内存 GiB | 环境安装秒 | 启动至首轮秒 |
|---|---|---|---:|---:|---:|
| L4 | NVIDIA L4 | Intel(R) Xeon(R) CPU @ 2.20GHz (12) | 53.0 | 145.6 | 106.8 |
| G4 | NVIDIA RTX PRO 6000 Blackwell Server Edition | AMD EPYC 9B45 (48) | 176.9 | 80.6 | 48.6 |
| A100 | NVIDIA A100-SXM4-40GB | Intel(R) Xeon(R) CPU @ 2.20GHz (12) | 83.5 | 141.7 | 94.7 |

这是单 seed 的端到端快速测量；CPU、宿主机和显存配置也影响吞吐，不能将差异全部归因于 GPU。显存每 500ms 采样，可能漏掉瞬时峰值。未测棋力、长期收敛或多 seed 方差。

L4 / G4 通过 `colab run` 执行此 Notebook 全部代码单元的逐字导出；A100 通过 `colab exec` 逐单元执行。下方保存的运行输出来自 A100，三卡完整指标另存于 Notebook 的 `benchmark_results` 元数据。

测试文件准备和本地校验完毕后才依次申请资源；每次测试结束后立即停止实例，并核实云端无活动实例。最终活动实例 0、消耗率 0.00/小时；余额从 129.27 降至 128.22 计算单位。未保存模型或 checkpoint；临时训练日志与环境已清理。手动在 Colab 重跑时，结束后请断开连接并删除 runtime。

In [ ]:
# @title 1. 固定训练参数
import csv
import json
import math
import os
from pathlib import Path
import signal
import subprocess
import sys
import tempfile
import time

REPO_URL = "https://github.com/GentleKingson/doudizhu.git"
COMMIT = "acdcf199cf422e2b683cb22142b8e7cb33b14460"
ITERATIONS = 40
WARMUP = 8
COLLECT = 224
ROLLOUT = 256
MINI_BATCHES = 3
SEED = 0
TRAIN_TIMEOUT_SECONDS = 600
REQUESTED_GPU = os.environ.get("REQUESTED_GPU", "manual")
os.environ.update(OMP_NUM_THREADS="6", WANDB_DISABLED="true",
                  PYTORCH_ALLOC_CONF="expandable_segments:True")
assert ITERATIONS >= WARMUP + 3


### 环境安装
安装和 C++ 编译耗时单独记录，不计入训练吞吐。环境放在临时目录，继承 Colab 的 CUDA PyTorch；不挂载 Drive。

In [ ]:
# @title 2. 安装仓库与 C++ 环境（不计入训练吞吐）
# 继承 Colab 的 CUDA PyTorch，避免重复下载 GPU 库；记录实际版本供比较。
workspace = tempfile.TemporaryDirectory(prefix="doudizhu-colab-", dir="/content")
work = Path(workspace.name)
repo = work / "repo"
python = work / "venv/bin/python"
setup_started = time.monotonic()
try:
    with (work / "setup.log").open("w") as setup_log:
        def setup_command(command, **kwargs):
            subprocess.run(command, check=True, stdout=setup_log,
                           stderr=subprocess.STDOUT, timeout=600, **kwargs)
        setup_command(["git", "clone", "--quiet", "--no-checkout", REPO_URL, str(repo)])
        setup_command(["git", "checkout", "--quiet", COMMIT], cwd=repo)
        assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() == COMMIT
        setup_command([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(work / "venv")])
        setup_command([str(python), "-m", "pip", "install", "--disable-pip-version-check",
                       "setuptools>=64", "wheel", "cmake==3.31.10", "ninja",
                       "numpy==2.5.3", "tyro==1.0.16", "tqdm"])
        setup_command([str(python), "-m", "pip", "install", "--no-deps", "-e", str(repo)])
        build_env = dict(os.environ, NANOPOKERLIB_BUILD_JOBS=str(min(8, os.cpu_count() or 2)))
        setup_command([str(python), "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(repo / "nanopokerlib")], env=build_env)
    setup_seconds = time.monotonic() - setup_started
    print(f"环境安装完成：{setup_seconds:.1f} 秒；仓库 {COMMIT[:12]}", flush=True)
except BaseException:
    print((work / "setup.log").read_text()[-4000:])
    workspace.cleanup()
    raise


环境安装完成：141.7 秒；仓库 acdcf199cf42


In [ ]:
# @title 3. 核对实际 GPU 与环境
hardware_code = """
import json, os, platform, subprocess, torch
assert torch.cuda.is_available(), '请先选择 GPU runtime'
assert torch.cuda.device_count() == 1, '本测试使用单 GPU'
cpu = next(line.split(':', 1)[1].strip() for line in open('/proc/cpuinfo') if line.startswith('model name'))
ram_kib = int(next(line.split()[1] for line in open('/proc/meminfo') if line.startswith('MemTotal:')))
print(json.dumps(dict(gpu=torch.cuda.get_device_name(0), cpu=cpu, cpu_count=os.cpu_count(),
                     cpu_affinity_count=len(os.sched_getaffinity(0)), ram_gib=ram_kib/1024**2,
                     python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda,
                     driver=subprocess.check_output(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],text=True).strip())))
"""
try:
    hardware = json.loads(subprocess.check_output([str(python), "-c", hardware_code], text=True))
    if REQUESTED_GPU in ("L4", "A100"):
        assert REQUESTED_GPU in hardware["gpu"], hardware
    print(json.dumps(hardware, indent=2, ensure_ascii=False))
except BaseException:
    workspace.cleanup()
    raise


{
  "gpu": "NVIDIA A100-SXM4-40GB",
  "cpu": "Intel(R) Xeon(R) CPU @ 2.20GHz",
  "cpu_count": 12,
  "cpu_affinity_count": 12,
  "ram_gib": 83.47369384765625,
  "python": "3.13.15",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "driver": "580.82.07"
}


### 测试
通过仓库已有的 JSONL 指标从外部观察完整采样、前向、反向及优化器时间，训练源码不变。保留基础指标，关闭每轮参数范数与对手评估。

In [ ]:
# @title 4. 计量逻辑与最小自检
def summarize(events, collect, warmup):
    measured = events[warmup:]
    elapsed = measured[-1]["seconds"] - measured[0]["seconds"]
    intervals = len(measured) - 1
    assert intervals > 0 and elapsed > 0
    steps = measured[-1]["statistics/game_steps_total"] - measured[0]["statistics/game_steps_total"]
    return dict(steady_seconds=elapsed, measured_iterations=intervals,
                measured_games=collect * intervals, games_per_second=collect * intervals / elapsed,
                game_steps_per_second=steps / elapsed, seconds_per_iteration=elapsed / intervals,
                first_iteration_seconds=events[0]["seconds"])

example = [dict(seconds=t, **{"statistics/game_steps_total":t*10}) for t in (10,12,14,16)]
assert summarize(example, 32, 1)["games_per_second"] == 16
assert summarize(example, 32, 1)["game_steps_per_second"] == 10
print("PASS：排除启动和预热，并按完整对局数归一化")


PASS：排除启动和预热，并按完整对局数归一化


In [ ]:
# @title 5. 运行 IPPO + Muon 快速测试（不保存 checkpoint）
events, pending, offset = [], "", 0
metrics = work / "metrics.jsonl"
gpu_path = work / "gpu.csv"
env = dict(os.environ, MARVEL_METRICS_JSONL=str(metrics), PYTHONUNBUFFERED="1")
command = [str(python), "-m", "marvel.train", "--algo-name", "ippo", "--game-name", "doudizhu",
           "--device", "cuda", "--seed", str(SEED), "--total-iterations", str(ITERATIONS),
           "--rollout-batch-size", str(ROLLOUT), "--collect-batch-size", str(COLLECT),
           "--mini-batches", str(MINI_BATCHES), "--compile-backend", "inductor",
           "--param-stats-interval", "-1", "--checkpoint-interval", "-1",
           "--evaluation-interval", "-1", "--logging-dir", str(work / "training")]
training = monitor = None
started = time.monotonic()
try:
    with gpu_path.open("w") as gpu_file, (work / "training.log").open("w") as log_file:
        monitor = subprocess.Popen(["nvidia-smi", "--id=0", "--query-gpu=memory.used,utilization.gpu",
                                    "--format=csv,noheader,nounits", "--loop-ms=500"],
                                   stdout=gpu_file, stderr=subprocess.DEVNULL)
        training = subprocess.Popen(command, cwd=repo, env=env, stdout=log_file,
                                    stderr=subprocess.STDOUT, start_new_session=True)
        def consume_metrics():
            global pending, offset
            if not metrics.exists():
                return
            with metrics.open() as fh:
                fh.seek(offset)
                pending += fh.read()
                offset = fh.tell()
            lines = pending.split("\n")
            pending = lines.pop()
            for line in lines:
                row = json.loads(line)
                assert all(math.isfinite(v) for v in row.values() if isinstance(v,(int,float)))
                if "statistics/game_steps_total" in row:
                    # shortcut: 20ms 外部轮询记录轮末时间；细分阶段耗时需内部计时。
                    row["seconds"] = time.monotonic() - started
                    events.append(row)
                    if len(events) in (1, 9, 20, ITERATIONS):
                        print(f"已完成 {len(events)}/{ITERATIONS} 轮", flush=True)
        while training.poll() is None:
            consume_metrics()
            if time.monotonic() - started > TRAIN_TIMEOUT_SECONDS:
                raise TimeoutError("训练测试超过时间上限")
            time.sleep(0.02)
        consume_metrics()  # 读取退出前写入的最后一轮，避免丢失最后一个指标。
    if training.returncode != 0:
        raise RuntimeError((work / "training.log").read_text()[-4000:])
    assert len(events) == ITERATIONS and [r["step"] for r in events] == list(range(1, ITERATIONS + 1))
    assert not list(work.rglob("*.pt")), "测试不应写入模型 checkpoint"
    monitor.terminate()
    monitor.wait(timeout=10)
    samples = list(csv.reader(gpu_path.read_text().splitlines()))
    result = dict(requested_gpu=REQUESTED_GPU, **hardware, repository_commit=COMMIT,
                  config=dict(algo="ippo", optimizer="Muon", model="v1.3", seed=SEED,
                              rollout=ROLLOUT, collect=COLLECT, mini_batches=MINI_BATCHES,
                              ppo_epochs=4, omp_threads=6, compile_backend="inductor",
                              allocator="expandable_segments:True", iterations=ITERATIONS, warmup=WARMUP),
                  setup_seconds=setup_seconds, wall_seconds=time.monotonic() - started,
                  gpu_peak_mib=max(float(r[0]) for r in samples), checkpoints_saved=0,
                  **summarize(events, COLLECT, WARMUP))
    print("COLAB_BENCHMARK_RESULT=" + json.dumps(result, ensure_ascii=False), flush=True)
finally:
    if training is not None and training.poll() is None:
        os.killpg(training.pid, signal.SIGTERM)
        try:
            training.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(training.pid, signal.SIGKILL)
            training.wait()
    if monitor is not None and monitor.poll() is None:
        monitor.terminate()
        monitor.wait(timeout=10)
    workspace.cleanup()


已完成 1/40 轮


已完成 9/40 轮


已完成 20/40 轮


已完成 40/40 轮


COLAB_BENCHMARK_RESULT={"requested_gpu": "A100", "gpu": "NVIDIA A100-SXM4-40GB", "cpu": "Intel(R) Xeon(R) CPU @ 2.20GHz", "cpu_count": 12, "cpu_affinity_count": 12, "ram_gib": 83.47369384765625, "python": "3.13.15", "torch": "2.11.0+cu130", "cuda": "13.0", "driver": "580.82.07", "repository_commit": "acdcf199cf422e2b683cb22142b8e7cb33b14460", "config": {"algo": "ippo", "optimizer": "Muon", "model": "v1.3", "seed": 0, "rollout": 256, "collect": 224, "mini_batches": 3, "ppo_epochs": 4, "omp_threads": 6, "compile_backend": "inductor", "allocator": "expandable_segments:True", "iterations": 40, "warmup": 8}, "setup_seconds": 141.74164569599998, "wall_seconds": 185.72468766100008, "gpu_peak_mib": 12646.0, "checkpoints_saved": 0, "steady_seconds": 60.99635915400006, "measured_iterations": 31, "measured_games": 6944, "games_per_second": 113.84286039873614, "game_steps_per_second": 7065.667623077088, "seconds_per_iteration": 1.9676244888387115, "first_iteration_seconds": 94.68290018699997}


### 结果
主指标为完成并用于更新的对局/秒；游戏步/秒用于辅助理解局长差异。启动至首轮时间包含编译和首轮，不能单独解释为编译时间。测试后立即释放 runtime，以停止计费。

In [ ]:
# @title 6. 测试结果
print(f"GPU：{result['gpu']}")
print(f"稳态吞吐：{result['games_per_second']:.2f} 对局/秒，{result['game_steps_per_second']:.0f} 游戏步/秒")
print(f"每轮耗时：{result['seconds_per_iteration']:.3f} 秒；显存观察峰值：{result['gpu_peak_mib']/1024:.2f} GiB")
print(f"启动至第一轮：{result['first_iteration_seconds']:.1f} 秒（含初始化、编译及首轮）")
print("未保存模型；临时训练日志与环境已清理。结束后请选择断开连接并删除 runtime。")


GPU：NVIDIA A100-SXM4-40GB
稳态吞吐：113.84 对局/秒，7066 游戏步/秒
每轮耗时：1.968 秒；显存观察峰值：12.35 GiB
启动至第一轮：94.7 秒（含初始化、编译及首轮）
未保存模型；临时训练日志与环境已清理。结束后请选择断开连接并删除 runtime。
